## Azure Firewall Basic — 1操作ずつ学ぶ実機ハンズオン

**iPhone版:** [スマートフォン向け学習画面を開く](/voila/render/mobile_learning.ipynb)。コマンドと結果を1操作ずつ確認できます。

操作の中身を隠さない学習用Notebookです。原則1コードセルにつき1つのAzure CLIコマンドを実行します。
コマンドは実行直前に、指定したサブスクリプションIDを含めて表示されます。

- 無料の準備: リソースグループ・VNet・Firewall Policy
- **有料:** Public IP、Azure Firewall Basic、Ubuntu VM
- 通信経路: ルートテーブルをVMサブネットに適用
- 通信の検査: GoogleとMicrosoftへのHTTPSを比較
- 最後に必ず学習リソースを削除

**警告:** Notebookを閉じてもAzure料金は止まりません。Notebookの「すべて実行」は使わないでください。
変更・削除コマンドの直前には、共通の安全ガードがAzure CLIでRGの管理タグと最新リソース一覧を確認し、
実行した追加の読み取りコマンドも出力します。これらは学習上の1操作に対する事前確認です。

## 今回の構成図（draw.ioで編集可能）

以下は**構築予定の構成**です。Azure内のリソースとインターネット上の通信先のみを記載しています。

![Azure Firewall Basic 学習用ネットワーク構成図](diagrams/azure-firewall-basic.png)

[編集可能なdraw.io図を開く](diagrams/azure-firewall-basic.drawio)

> Firewall Policy、Public IP、UDRは独立したAzureリソースであり、サブネットの内部に配置されるものではありません。緑/赤の経路はGoogle許可ルール適用後の**期待動作**を示します。実測済みではありません。

### 0-1. Notebookの実行セッションを用意

Python補助機能を読み込むだけでAzure操作は行いません。

このセルはAzure CLIの実行ではなく、Notebook内の準備や選択を行います。

In [ ]:
from notebook_helpers import LearningSession
session = LearningSession()
print('Azure CLIをまだ実行していません')

### 0-2. Azureにサインイン

デバイスコード認証用のコードとURLが表示されます。iPhone Safariで認証します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.login()

### 0-3. Azureサブスクリプション一覧を確認

請求先になる契約名とIDを確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
accounts = session.run("az account list --all", scoped=False)

### 0-4. Azureサブスクリプションを選択

以降のコマンドにはサブスクリプションIDが自動で付加されます。

このセルはAzure CLIの実行ではなく、Notebook内の準備や選択を行います。

In [ ]:
session.select_subscription(accounts)

## 1. リソースグループ

今回の学習は専用のrg-firewall-labを利用します。以前の学習で空のグループが作成済みですが、必ず現在の状態を確認します。

### 1-1. RGの状態を読む

存在しない場合はエラーになります。その場合のみ次の新規作成を行います。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az group show -n rg-firewall-lab')

### 1-2（存在しない場合のみ）. RGを新規作成

既存のRGがある場合は実行しません。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az group create -n rg-firewall-lab -l japaneast --tags firewallLab=azure-firewall-mobile-handson-v1')

### 1-3. RG内のリソースを確認

空でない場合は、教材外のリソースを変更しないためここで中止します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
existing = session.run("az resource list -g rg-firewall-lab")
session.expect_group_empty(existing)

### 1-4. RGに教材用タグを付ける

操作画面でも同じタグを使い、安全のため教材のRGを識別します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az group update -n rg-firewall-lab --set tags.firewallLab=azure-firewall-mobile-handson-v1')

### 1-5. タグが反映されたか再取得

単に更新コマンドが成功しただけでなく、実際のタグを読みます。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
group = session.run("az group show -n rg-firewall-lab")
assert group["tags"]["firewallLab"] == "azure-firewall-mobile-handson-v1"

## 2. ネットワーク — VNetと3サブネット

VNetは10.0.0.0/16です。次の3サブネットを別の操作として作ります。

- AzureFirewallSubnet：10.0.0.0/26 （Firewall本体）
- AzureFirewallManagementSubnet：10.0.1.0/26 （管理）
- WorkloadSubnet：10.0.2.0/24 （VM）

※ VNetの最初の作成コマンドでは、最初のサブネットも同時に作られます。

### 2-1. VNet + Firewallサブネットを作成

最初のAzureFirewallSubnetはFirewall本体の配置先です。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet create -g rg-firewall-lab -n vnet-firewall-lab -l japaneast --address-prefix 10.0.0.0/16 --subnet-name AzureFirewallSubnet --subnet-prefix 10.0.0.0/26')

### 2-2. 管理サブネットを追加

Basic SKUで必要な管理用の独立したサブネットです。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet subnet create -g rg-firewall-lab --vnet-name vnet-firewall-lab -n AzureFirewallManagementSubnet --address-prefix 10.0.1.0/26')

### 2-3. VMサブネットを追加

通信制御の対象になるVMを配置します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet subnet create -g rg-firewall-lab --vnet-name vnet-firewall-lab -n WorkloadSubnet --address-prefix 10.0.2.0/24')

### 2-4. サブネットを確認

3つの名前とIPアドレス範囲が期待どおりか見ます。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet subnet list -g rg-firewall-lab --vnet-name vnet-firewall-lab')

## 3. Firewall Policy Basic

Policyはルールを管理する独立したリソースです。Firewall本体をまだ作っていないため、これだけでは通信制御されません。

### 3-1. Policyを作成

Firewall PolicyのSKUをBasicにします。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy create -g rg-firewall-lab -n policy-firewall-lab -l japaneast --sku Basic')

### 3-2. Policyを確認

AzureからBasic SKUの実際の設定を読み返します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy show -g rg-firewall-lab -n policy-firewall-lab')

## 4. 有料リソース — Firewall本体

以下は課金を伴います。Public IPとAzure Firewallは、作成後に削除するまで課金されます。
課金に関するチェックと確認文言を入れるまで、有料コマンドは実行しません。

### 4-0. 課金の確認

画面でチェックを入れ、CREATE BILLABLEと入力します。まだAzure変更はありません。

このセルはAzure CLIの実行ではなく、Notebook内の準備や選択を行います。

In [ ]:
session.approvals()

### 4-1. 通信用Public IPを作る

Firewall本体のデータプレーン用Standard Public IPです。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network public-ip create -g rg-firewall-lab -n pip-firewall-lab -l japaneast --sku Standard --allocation-method Static', paid=True)

### 4-2. 管理用Public IPを作る

Firewall Basicの管理通信を別のIPで扱います。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network public-ip create -g rg-firewall-lab -n pip-firewall-mgmt-lab -l japaneast --sku Standard --allocation-method Static', paid=True)

### 4-3. Public IPを確認

2つのStandard Public IPが作成されたか確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network public-ip list -g rg-firewall-lab')

### 4-4. Firewall Basicを作成

VNet、2つのPublic IP、Policyを関連付けます。Firewallの時間課金が始まります。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall create -g rg-firewall-lab -n fw-firewall-lab -l japaneast --sku AZFW_VNet --tier Basic --vnet-name vnet-firewall-lab --conf-name fw-ipconfig --public-ip pip-firewall-lab --m-conf-name fw-mgmtconfig --m-public-ip pip-firewall-mgmt-lab --firewall-policy policy-firewall-lab', paid=True)

### 4-5. Firewallの状態とPrivate IPを確認

Private IPはUDRに使うため控えます。SKU・Policy・IPを画面で確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
fw = session.run("az network firewall show -g rg-firewall-lab -n fw-firewall-lab")
fw_ip = fw["ipConfigurations"][0]["privateIPAddress"]
print("Firewall Private IP:", fw_ip)

## 5. Ubuntu VM

VMはWorkloadSubnet内に置き、Public IPは割り当てません。Run CommandでVM内部からcurlを実行します。

### 5-1. VM用NICを作成

NICとサブネットの関係がVMの通信経路に関係します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network nic create -g rg-firewall-lab -n nic-firewall-lab -l japaneast --vnet-name vnet-firewall-lab --subnet WorkloadSubnet', paid=True)

### 5-2. NICにPublic IPが付いていないことを確認

NICのIP構成とsubnet参照を読み取ります。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network nic show -g rg-firewall-lab -n nic-firewall-lab')

### 5-3. Ubuntu VMを作成

Standard_B1sのUbuntu 22.04 VMを作ります。ディスクも課金対象です。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az vm create -g rg-firewall-lab -n vm-firewall-lab -l japaneast --nics nic-firewall-lab --image Ubuntu2204 --size Standard_B1s --storage-sku Standard_LRS --os-disk-name osdisk-firewall-lab --admin-username azureuser --generate-ssh-keys', paid=True)

### 5-4. VMを確認

Azureに記録されたVMのサイズとネットワーク参照を確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az vm show -g rg-firewall-lab -n vm-firewall-lab')

## 6. UDR — なぜFirewallへ通信が流れるのか

Firewallが存在するだけではVMの通信は通りません。ルートテーブル・既定ルート・サブネット関連付けの3つの変更を分けます。

### 6-1. ルートテーブルを作る

まだどのサブネットにも適用されない入れ物を作成します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network route-table create -g rg-firewall-lab -n rt-firewall-lab -l japaneast', paid=True)

### 6-2. 既定ルートを作る

0.0.0.0/0の転送先をFirewallのPrivate IPに指定します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
assert fw_ip
session.run(f"az network route-table route create -g rg-firewall-lab --route-table-name rt-firewall-lab -n default-via-firewall --address-prefix 0.0.0.0/0 --next-hop-type VirtualAppliance --next-hop-ip-address {fw_ip}", paid=True)

### 6-3. ルートを読み返す

nextHopTypeとIPを確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network route-table route show -g rg-firewall-lab --route-table-name rt-firewall-lab -n default-via-firewall')

### 6-4. VMサブネットへルートテーブルを関連付ける

**この操作から**WorkloadSubnetの外向き通信がFirewallへ向かいます。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet subnet update -g rg-firewall-lab --vnet-name vnet-firewall-lab -n WorkloadSubnet --route-table rt-firewall-lab', paid=True)

### 6-5. サブネットの設定を読み返す

関連付けたルートテーブルのIDを確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network vnet subnet show -g rg-firewall-lab --vnet-name vnet-firewall-lab -n WorkloadSubnet')

### 6-6. NICの有効経路を確認

NICから見て0.0.0.0/0がVirtualApplianceへ向いていることを調べます。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network nic show-effective-route-table -g rg-firewall-lab -n nic-firewall-lab')

## 7. ルールなしで通信試験

Firewall Policyに許可ルールがない状態で、GoogleとMicrosoftへのHTTPS結果を測ります。
curlの失敗だけではFirewall遮断と断定できません。HTTP 403/500でもHTTP応答が届いているため遮断ではありません。
DNS失敗（curl 6）、TLS証明書エラー（60）、タイムアウト（28）、接続失敗（7）を区別します。
また、Firewall経由のAzure宛送信が遮断されるとVM Run Commandが使えなくなる可能性があります。
この場合はFirewallの拒否試験が成功したと判定せず、VMエージェント用通信の許可や別の観測方法を検討してください。

### 7-1. ルールグループ一覧を確認

ルールがない前提であることを確かめます。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy rule-collection-group list -g rg-firewall-lab --policy-name policy-firewall-lab')

### 7-2. VM内のcurlで2ホストをテスト

DNS・終了コード・HTTPステータスを分けて表示します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
import shlex
probe = "for host in www.google.com www.microsoft.com; do if getent ahostsv4 $host >/dev/null 2>&1; then dns=ok; else dns=fail; fi; code=$(curl -I -sS -o /dev/null -w '%{http_code}' --connect-timeout 10 --max-time 20 https://$host); rc=$?; echo RESULT host=$host dns=$dns exit=$rc http=$code; done"
before = session.run("az vm run-command invoke -g rg-firewall-lab -n vm-firewall-lab --command-id RunShellScript --scripts " + shlex.quote(probe), paid=True)

## 8. GoogleへのHTTPS通信だけ許可

Rule Collection Groupを作り、その中にApplication Ruleを追加します。2つのAzure CLIコマンドに分けます。

### 8-1. Rule Collection Groupを作成

ルール集合を置く優先順位200のグループです。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy rule-collection-group create -g rg-firewall-lab --policy-name policy-firewall-lab -n rcg-web-lab --priority 200', paid=True)

### 8-2. Googleだけ許可するルールを追加

WorkloadSubnetからwww.google.comへのHTTPS 443に限定します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy rule-collection-group collection add-filter-collection -g rg-firewall-lab --policy-name policy-firewall-lab --rule-collection-group-name rcg-web-lab --name allow-google --collection-priority 100 --action Allow --rule-name google-https --rule-type ApplicationRule --source-addresses 10.0.2.0/24 --protocols Https=443 --target-fqdns www.google.com', paid=True)

### 8-3. Policyのルールを再取得

許可する送信元、FQDN、プロトコル、優先順位を確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run('az network firewall policy rule-collection-group show -g rg-firewall-lab --policy-name policy-firewall-lab -n rcg-web-lab')

### 8-4. curlを再実行して比較

同じコマンドでGoogle成功、Microsoft拒否となることを観察します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
after = session.run("az vm run-command invoke -g rg-firewall-lab -n vm-firewall-lab --command-id RunShellScript --scripts " + shlex.quote(probe), paid=True)

## 9. 学習の理解を確認

1. Policyを作成しただけでは、なぜ通信制御が始まらないのでしょうか？
2. 既定ルート0.0.0.0/0とは、何を表しているでしょうか？
3. GoogleとMicrosoftで結果が異なる理由を、Firewall PolicyとApplication Ruleの言葉で説明できますか？

**検証上の注意:** Firewall診断ログは別途Azure Monitorの設定を要します。
このNotebookでは自動構築していないため、単にcurlが失敗しただけではFirewallによる拒否を証明できません。

## 10. 学習終了とAzureリソースの削除（必須）

Firewall/VM/Public IPは削除完了まで料金が続きます。削除前にグループ内の実際のリソースを確認します。

### 10-1. 削除対象を一覧表示

教材以外のリソースが混ざった場合は中止してください。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
inventory = session.run("az resource list -g rg-firewall-lab")
print("削除候補:", [(r.get("name"), r.get("type")) for r in inventory])

### 10-2. 教材用タグを確認

教材用のタグがあることを確認します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
owned = session.run("az group show -n rg-firewall-lab")
if (owned.get("tags") or {}).get("firewallLab") != "azure-firewall-mobile-handson-v1":
    raise RuntimeError("管理タグが一致しません")

### 10-3. 削除承認を入力

DELETE rg-firewall-labと入力するまでは削除できません。Azureの変更はありません。

このセルはAzure CLIの実行ではなく、Notebook内の準備や選択を行います。

In [ ]:
session.cleanup_approval()

### 10-4. リソースグループを削除

元に戻せません。実行直前に**共通の安全確認**として管理タグ・現在のリソース一覧をAzureから再取得し、教材外のリソースがあれば削除を停止します。実行した確認CLIも出力に表示します。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
session.run("az group delete -n rg-firewall-lab --yes", delete=True)

### 10-5. 削除完了をAzureから確認

falseならグループが存在しません。

次のセルは、記載のAzure CLIコマンドを1つだけ実行し、結果を同じセルに表示します。

In [ ]:
exists = session.run("az group exists -n rg-firewall-lab")
session.verify_deleted(exists)

## 復習と完了判定

各セルで行ったCLI操作とAzureの応答を照合し、どの操作がどんな変更を生んだか振り返ってください。実Azureの動作証拠が揃わなければ、模擬検証や画面表示の成功だけで完了とは判定しません。